# Cross Encoder testing
Test the performance of cross encoder in pairing soilvoc keywords with record metadata (title, abstract, pdf, ...)

In [2]:
import json, time
import torch
from sentence_transformers import CrossEncoder

t0 = time.time()

DOC = """Relative Contribution Of Trees And Crops To Soil Carbon Content In A Parkland System In Burkina Faso Using Variations In Natural C-13 Abundance","The Origin Of Organic Matter Was Studied In The Soils Of A Parkland Of Karite (Vitallaria Paradoxa C.F. Gaertn) And Nere (Parkia Biglobosa (Jacq.) Benth.), Which Is Extensively Cultivated Without The Use Of Fertilisers. In Such Systems, Fertility (Physical, Chemical And Biological) Gradients Around Trees Have Been Attributed By Some Authors To A Priori Differences In Fertility, Allowing For Better Tree Establishment On Richer Sites. In Reverse, Other Workers Believed That These Gradients Are Due To The Contribution Of Trees To The Formation Of Soil Organic Matter Through Litter And Decay Of Roots. Measurements Of The Variations In The C-13 Isotopic Composition Allowed For A Distinction Between Tree (C-3) Derived C And Crop And Grass (C-4) Derived C In The Total Soil Organic C Content. The Organic Carbon Contents Of The Soils Were Recorded Under The Two Species At Two Soil Depths And At Five Distances Going From Tree Trunk To The Open Area And Their C Isotopic Signatures Were Analysed. The Results Showed That Soil Carbon Contents Under Karite (6.43 +/- 0.45 G Kg(-1)) And Nere (5.65 +/- 0.27 G Kg(-1)) Were Significantly Higher (P < 0.01) Than In The Open Area (4.09 +/- 0.26 G Kg(-1)). The Delta C-13 Of Soil C Was Significantly Higher (P < 0.001) In The Open Area (-17.5 +/- 0.3 Parts Per Thousand) Compared With The Values Obtained On Average With Depth And Distance From Tree Under Karite (-20.2 +/- 0.4 Parts Per Thousand) And Nere (-20.1 +/- 0.4 Parts Per Thousand). The C-4-Derived Soil C Was Approximately Constant, And The Differences In Total Soil C Were Fully Explained By The C-3 (Tree) Contributions To Soil Carbon Of 4.01 +/- 0.71, 3.02 +/- 0.53, 1.53 +/- 0.10 G Kg(-1), Respectively Under Karite, Nere And In The Open Area. These Results Show That Trees In Parklands Have A Directly Positive Contribution To Soil Carbon Content, Justifying The Need To Encourage The Maintenance Of Trees In These Systems In Semi-Arid Environments Where The Carbon Content Of Soil Appears To Be The First Limiting Factor For Crop Growth.
"""

with open("../concepts_multilingual.json", encoding="utf-8") as f:
    concepts = json.load(f)

# One (label, concept) entry per English label; several labels share a concept.
labels = [(lab, c["identifier"].split("#")[-1])
          for c in concepts for lab in c["labels"].get("en", [])]
print(f"{len(labels)} English labels from {len(concepts)} concepts")

model = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
                     activation_fn=torch.nn.Sigmoid(), max_length=256)
scores = model.predict([(lab, DOC) for lab, _ in labels], show_progress_bar=True)
for score, (lab, cid) in sorted(zip(scores, labels), reverse=True)[:10]:
    print(f"  {score:.4f}  {lab:<32} {cid}")

print(f"\ntotal {time.time()-t0:.1f}s")



1064 English labels from 799 concepts


Batches: 100%|██████████| 34/34 [01:29<00:00,  2.64s/it]

  0.8791  soil organic matter content      SoilOrganicMatterContents
  0.8339  soil organic matter contents     SoilOrganicMatterContents
  0.7613  soil organic matter              SoilOrganicMatter
  0.7025  soil organic carbon              SoilOrganicCarbon
  0.4383  soil organic components          SoilOrganicComponents
  0.4254  soil organic component           SoilOrganicComponents
  0.2185  critical soil organic matter content CriticalSoilOrganicMatterContents
  0.1831  soil inorganic carbon            SoilInorganicCarbon
  0.1815  soil organic matter class        SoilOrganicMatterClass
  0.1516  critical soil organic matter contents CriticalSoilOrganicMatterContents

total 93.0s


In [8]:
# Now try to do with the german content.
DOC_DE = """
Die Gesamt-Phosphoreinträge in die Gewässer wurden mit dem Stoffflussmodell MODIFFUS über alle diffusen Eintragsquellen (Ackerland, Dauergrünland, Wald, Gletscher, Siedlungsgrünflächen etc.) und alle diffusen Eintragspfade (Bodenerosion, Auswaschung, Abschwemmung, Drainage, atmosphärische Deposition etc.) berechnet. Die Karte zeigt die aufsummierten Verluste pro Landnutzungskategorie im Hektarraster, basierend auf der Arealstatistik 2013/18. Es wurden mittlere klimatische Bedingungen zugrunde gelegt, das Bezugsjahr ist 2020.
"""
scores = model.predict([(lab, DOC_DE) for lab, _ in labels], show_progress_bar=True)

for score, (lab, cid) in sorted(zip(scores, labels), reverse=True)[:10]:
    print(f"  {score:.4f}  {lab:<32} {cid}")

Batches: 100%|██████████| 34/34 [00:49<00:00,  1.46s/it]

  0.8466  phosphorus total elements        PhosphorusTotalElements
  0.6913  soil phosphorus loss             SoilPLoss
  0.5186  soil water loss                  SoilWaterLoss
  0.4520  soil P loss                      SoilPLoss
  0.4472  soil water deficit               SoilMoistureDeficit
  0.4143  soil particle movement           SoilParticleMovement
  0.4046  land use class                   LandUseClass
  0.3834  soil organic carbon loss         SOCLoss
  0.3286  soil water contents              SoilWaterContents
  0.3022  soil oxygen contents             SoilOxygenContents


CE doing better than KeyBERT in pairing content with keywords in different languages (de - en). The result looks ok, but Phosphorus (P) is not there.

In [ ]:
# "Bi-encoder retrieval + cross-encoder rerank
# set up
import re
import numpy as np
from sentence_transformers import SentenceTransformer

CONCEPTS_PATH = "../concepts_multilingual.json"
BI_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
CE_MODEL = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"


def load_vocab(path=CONCEPTS_PATH, lang="en"):
    """-> (labels, concept_ids): one entry per label; a concept may have several."""
    with open(path, encoding="utf-8") as f:
        concepts = json.load(f)
    pairs = [(lab, c["identifier"].split("#")[-1])
             for c in concepts for lab in c["labels"].get(lang, [])]
    labels, concept_ids = map(list, zip(*pairs))
    print(f"{len(labels)} {lang} labels from {len(concepts)} concepts")
    return labels, concept_ids


LABELS, CONCEPT_IDS = load_vocab()
bi = SentenceTransformer(BI_MODEL)
ce = CrossEncoder(CE_MODEL, activation_fn=torch.nn.Sigmoid(), max_length=256)
LABEL_EMB = bi.encode(LABELS, batch_size=128, normalize_embeddings=True,
                      show_progress_bar=True)   # 1064 x 384, reused for every doc

def chunk_text(text, lo=200, hi=900):
    """Sentence-split, then merge back into blocks of roughly lo..hi characters."""
    text = re.sub(r"\s+", " ", text).strip()
    chunks, buf = [], ""
    for sent in re.split(r"(?<=[.!?])\s+(?=[A-ZÄÖÜ])", text):
        if buf and len(buf) + len(sent) + 1 > hi:
            chunks.append(buf)
            buf = sent
        else:
            buf = f"{buf} {sent}".strip()
        if len(buf) >= lo:
            chunks.append(buf)
            buf = ""
    if buf:                      # trailing fragment joins the last chunk
        if chunks and len(buf) < lo:
            chunks[-1] += " " + buf
        else:
            chunks.append(buf)
    return chunks


def retrieve(chunks, top_k=20):
    """Stage 1 — bi-encoder shortlist: top_k labels per chunk.
    -> (pairs, cos) where pairs is [(chunk_index, label_index), ...]"""
    cos = bi.encode(chunks, normalize_embeddings=True) @ LABEL_EMB.T
    k = min(top_k, len(LABELS) - 1)
    pairs = [(ci, int(li)) for ci in range(len(chunks))
             for li in np.argpartition(-cos[ci], k)[:k]]
    return pairs, cos


def rerank(chunks, pairs, cos, batch_size=32, progress=False):
    """Stage 2 — cross-encoder scores each (label, chunk) pair, best kept per concept."""
    scores = ce.predict([(LABELS[li], chunks[ci]) for ci, li in pairs],
                        batch_size=batch_size, show_progress_bar=progress)
    best = {}
    for (ci, li), s in zip(pairs, scores):
        cid = CONCEPT_IDS[li]
        if float(s) > best.get(cid, {}).get("ce", -1):
            best[cid] = {"concept": cid, "label": LABELS[li], "ce": float(s),
                         "cos": float(cos[ci, li]), "chunk": ci}
    return sorted(best.values(), key=lambda r: -r["ce"])


def rank_concepts(doc, top_k=20, lo=200, hi=900, verbose=True):
    """Full pipeline for one record -> ranked list of dicts."""
    t0 = time.time()
    chunks = chunk_text(doc, lo, hi)
    pairs, cos = retrieve(chunks, top_k)
    ranked = rerank(chunks, pairs, cos, progress=verbose)
    if verbose:
        n_lab = len({li for _, li in pairs})
        print(f"{len(doc)} chars -> {len(chunks)} chunks | {len(pairs)} CE pairs, "
              f"{n_lab} labels = {n_lab / len(LABELS):.0%} of vocab | "
              f"{time.time() - t0:.1f}s")
    return ranked


def show(ranked, n=15):
    print(f"  {'ce':>7} {'cos':>6} {'chk':>3}  label")
    for r in ranked[:n]:
        print(f"  {r['ce']:7.4f} {r['cos']:6.3f} {r['chunk']:>3}  "
              f"{r['label']:<34} {r['concept']}")




1064 en labels from 799 concepts


Batches: 100%|██████████| 9/9 [00:01<00:00,  4.71it/s]


In [6]:

show(rank_concepts(DOC))                     

Batches: 100%|██████████| 5/5 [00:03<00:00,  1.64it/s]

2196 chars -> 8 chunks | 160 CE pairs, 92 labels = 9% of vocab | 3.4s
       ce    cos chk  label
   0.9939  0.749   3  soil organic carbon                SoilOrganicCarbon
   0.9769  0.616   3  soil organic matter content        SoilOrganicMatterContents
   0.9555  0.731   6  soil total carbon                  SoilTotalCarbon
   0.8216  0.693   4  soil carbon density                SoilCarbonDensity
   0.8028  0.627   3  soil organic component             SoilOrganicComponents
   0.7475  0.616   2  soil organic matter                SoilOrganicMatter
   0.6906  0.703   3  soil organic carbon loss           SOCLoss
   0.5926  0.720   3  soil inorganic carbon              SoilInorganicCarbon
   0.5749  0.607   3  critical soil organic matter content CriticalSoilOrganicMatterContents
   0.5098  0.483   1  fertiliser use                     FertiliserUse
   0.4877  0.606   2  soil organic matter class          SoilOrganicMatterClass
   0.3905  0.589   4  soil gravel content               

In [9]:
DOC_DE = """
Die Gesamt-Phosphoreinträge in die Gewässer wurden mit dem Stoffflussmodell MODIFFUS über alle diffusen Eintragsquellen (Ackerland, Dauergrünland, Wald, Gletscher, Siedlungsgrünflächen etc.) und alle diffusen Eintragspfade (Bodenerosion, Auswaschung, Abschwemmung, Drainage, atmosphärische Deposition etc.) berechnet. Die Karte zeigt die aufsummierten Verluste pro Landnutzungskategorie im Hektarraster, basierend auf der Arealstatistik 2013/18. Es wurden mittlere klimatische Bedingungen zugrunde gelegt, das Bezugsjahr ist 2020.
"""
show(rank_concepts(DOC_DE))  

Batches: 100%|██████████| 2/2 [00:01<00:00,  1.59it/s]

532 chars -> 2 chunks | 40 CE pairs, 40 labels = 4% of vocab | 1.4s
       ce    cos chk  label
   0.5546  0.420   1  land use class                     LandUseClass
   0.3380  0.630   0  soil water contents                SoilWaterContents
   0.2571  0.623   0  soil water flow                    SoilWaterFlow
   0.2232  0.632   0  soil water content                 SoilMoisture
   0.1505  0.616   0  soil water condensation            SoilWaterCondensation
   0.1465  0.419   1  SOC loss                           SOCLoss
   0.1211  0.625   0  water transfer (in soil)           SoilWaterMovement
   0.1085  0.616   0  soil water infiltration            SoilWaterInfiltration
   0.0831  0.628   0  soil water diffusivity             SoilWaterDiffusivity
   0.0636  0.438   1  soil P loss                        SoilPLoss
   0.0635  0.411   1  soil water loss                    SoilWaterLoss
   0.0594  0.625   0  soil mobile water contents         SoilMobileWaterContents
   0.0570  0.411   1  l

Summary: The retrieve-rerankng approach performs good on english content, bad on germany content. We can try to adjust the chunk_text process.